# Notebook 02 — 멀티모달 임베딩 & ChromaDB 인덱싱
## TechDocRAG: 멀티모달 기술 문서 RAG
### 이 노트북에서 배울 것
- BGE-M3 다국어 텍스트 임베딩 (한/영 동시 지원)
- BM25 키워드 인덱스 구축
- ChromaDB 컬렉션 설계 (텍스트 벡터 + 이미지 경로 메타데이터)
- 인덱싱 성능 벤치마크

In [ ]:
# 한글 폰트 + 기본 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os, sys, json, time
from pathlib import Path

ROOT       = Path().absolute().parent
OUTPUT_DIR = ROOT / 'output'
VECTOR_DIR = ROOT / 'vector_db'
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

# NB01에서 저장한 청크 로드
chunks_path = OUTPUT_DIR / 'chunks.json'
with open(chunks_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f"로드된 청크: {len(chunks)}개")
for c in chunks:
    print(f"  [{c['chunk_id']}] type={c['content_type']}, chars={c['char_count']}")

## 1. BGE-M3 텍스트 임베딩
BGE-M3는 한국어/영어/중국어 등 100개 언어를 동시 지원하는 SOTA 임베딩 모델입니다.
기술 문서에 한/영 혼용이 많아 최적의 선택입니다.

In [ ]:
# BGE-M3 임베딩 모델 로드
# 첫 실행 시 HuggingFace에서 ~570MB 다운로드
from sentence_transformers import SentenceTransformer
import numpy as np

print("BGE-M3 모델 로드 중... (첫 실행 시 ~570MB 다운로드)")
t0 = time.time()
embedder = SentenceTransformer('BAAI/bge-m3', cache_folder=str(ROOT / 'models'))
print(f"로드 완료: {time.time()-t0:.1f}초")
print(f"임베딩 차원: {embedder.get_sentence_embedding_dimension()}")

In [ ]:
# 청크 텍스트 임베딩 생성
# BGE-M3는 쿼리/패시지 구분 없이 동일 모델 사용 (symmetric)
texts = [c['text'] for c in chunks]

print(f"임베딩 생성 중... ({len(texts)}개 청크)")
t0 = time.time()
embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,   # 코사인 유사도 계산을 위한 L2 정규화
    show_progress_bar=True,
    batch_size=8
)
elapsed = time.time() - t0

print(f"\n임베딩 완료: {elapsed:.2f}초")
print(f"임베딩 shape: {embeddings.shape}  → ({len(texts)}개 청크, {embeddings.shape[1]}차원)")

# 정규화 확인 (norm ≈ 1.0)
norms = np.linalg.norm(embeddings, axis=1)
print(f"벡터 L2 norm: min={norms.min():.4f}, max={norms.max():.4f} (정규화됨 → ≈1.0)")

In [ ]:
# 청크 간 코사인 유사도 히트맵 시각화
import matplotlib.pyplot as plt

sim_matrix = embeddings @ embeddings.T  # 정규화된 벡터 → 내적 = 코사인 유사도

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(chunks)))
ax.set_yticks(range(len(chunks)))
ax.set_xticklabels([c['chunk_id'].split('_p')[-1] for c in chunks], rotation=45)
ax.set_yticklabels([c['chunk_id'].split('_p')[-1] for c in chunks])
ax.set_title('청크 간 코사인 유사도 (BGE-M3)', pad=12)
plt.colorbar(im, ax=ax, label='유사도')
plt.tight_layout()
plt.show()

# 가장 유사한 페이지 쌍 출력
for i in range(len(chunks)):
    for j in range(i+1, len(chunks)):
        print(f"  p{i} ↔ p{j}: {sim_matrix[i,j]:.3f}")

## 2. BM25 키워드 인덱스 구축
밀집 벡터 검색만으로는 정확한 키워드(DTC 코드, 모델명 등)를 놓칠 수 있습니다.
BM25와 앙상블하면 키워드 + 의미 검색 두 장점을 모두 얻습니다.

In [ ]:
# BM25 인덱스 구축
import re
from rank_bm25 import BM25Okapi

def tokenize_ko(text: str) -> list[str]:
    """간단한 한국어 토크나이저 — 공백 + 특수문자 분리"""
    text = text.lower()
    # 영문/숫자 단어 보존, 한글은 2글자 이상 유지
    tokens = re.findall(r'[a-z0-9]+|[가-힣]{2,}', text)
    return tokens

# 코퍼스 토크나이징
tokenized_corpus = [tokenize_ko(c['text']) for c in chunks]

# BM25 인덱스 생성
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 인덱스 구축 완료")
print(f"코퍼스 크기: {len(tokenized_corpus)}개 문서")
print(f"평균 토큰 수: {sum(len(t) for t in tokenized_corpus) / len(tokenized_corpus):.1f}")
print(f"\n토크나이징 예시 (페이지 0):")
print(f"  원문: {chunks[0]['text'][:80]}...")
print(f"  토큰: {tokenized_corpus[0][:20]}")

In [ ]:
# BM25 검색 테스트 — DTC 코드 같은 정확한 키워드 검색
test_queries = [
    "P0A1E 배터리 온도 과열",
    "SOC 칼만 필터 추정",
    "셀 밸런싱"
]

print("=== BM25 검색 테스트 ===")
for query in test_queries:
    q_tokens = tokenize_ko(query)
    scores   = bm25.get_scores(q_tokens)
    ranked   = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    print(f"\n쿼리: '{query}'")
    for rank, (idx, score) in enumerate(ranked[:3]):
        print(f"  #{rank+1} [{chunks[idx]['chunk_id']}] score={score:.3f}")

## 3. ChromaDB 인덱싱
텍스트 임베딩과 이미지 경로를 ChromaDB에 함께 저장합니다.
이미지는 파일 시스템에 두고 경로만 메타데이터로 관리합니다 (DB 용량 최적화).

In [ ]:
# ChromaDB 초기화 — 영구 저장 모드
import chromadb

client = chromadb.PersistentClient(path=str(VECTOR_DIR))

# 기존 컬렉션 초기화 (재실행 시 중복 방지)
try:
    client.delete_collection("tech_docs")
    print("기존 컬렉션 삭제 후 재생성")
except:
    pass

# 컬렉션 생성 — cosine 유사도 사용 (BGE-M3 정규화 벡터에 최적)
collection = client.create_collection(
    name="tech_docs",
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB 컬렉션 생성: 'tech_docs'")
print(f"저장 경로: {VECTOR_DIR}")

In [ ]:
# ChromaDB에 청크 + 임베딩 + 메타데이터 저장
t0 = time.time()

collection.add(
    ids        = [c['chunk_id'] for c in chunks],
    embeddings = embeddings.tolist(),
    documents  = [c['text'] for c in chunks],
    metadatas  = [{
        'doc_name':     c['doc_name'],
        'page_num':     c['page_num'],
        'image_path':   c['image_path'],
        'content_type': c['content_type'],
        'has_table':    str(c['has_table']),   # ChromaDB는 bool 미지원 → str
        'char_count':   c['char_count']
    } for c in chunks]
)

print(f"ChromaDB 인덱싱 완료: {time.time()-t0:.2f}초")
print(f"저장된 문서 수: {collection.count()}")

# 저장 확인
result = collection.get(ids=[chunks[0]['chunk_id']], include=['metadatas'])
print(f"\n[검증] 첫 번째 청크 메타데이터:")
for k, v in result['metadatas'][0].items():
    print(f"  {k}: {v}")

## 4. 하이브리드 검색 미리보기
ChromaDB 밀집 검색과 BM25 키워드 검색을 7:3으로 앙상블합니다.
NB03에서 정식 구현하고, 여기선 동작 원리를 확인합니다.

In [ ]:
# 하이브리드 검색 프리뷰 — Dense 70% + BM25 30%
def hybrid_search_preview(query: str, top_k: int = 3, dense_w: float = 0.7):
    # 1) Dense 검색 (ChromaDB)
    q_emb  = embedder.encode([query], normalize_embeddings=True)
    dense_res = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )
    # cosine distance → similarity (ChromaDB cosine space returns distance = 1 - sim)
    dense_scores = {
        dense_res['ids'][0][i]: 1 - dense_res['distances'][0][i]
        for i in range(len(dense_res['ids'][0]))
    }

    # 2) BM25 검색
    q_tokens   = tokenize_ko(query)
    bm25_raw   = bm25.get_scores(q_tokens)
    bm25_max   = max(bm25_raw) if max(bm25_raw) > 0 else 1
    bm25_norm  = {chunks[i]['chunk_id']: bm25_raw[i] / bm25_max for i in range(len(chunks))}

    # 3) 앙상블
    all_ids = set(dense_scores) | set(bm25_norm)
    hybrid  = {
        cid: dense_w * dense_scores.get(cid, 0) + (1 - dense_w) * bm25_norm.get(cid, 0)
        for cid in all_ids
    }
    ranked = sorted(hybrid.items(), key=lambda x: x[1], reverse=True)[:top_k]

    return ranked

# 테스트
test_cases = [
    ("배터리 온도 과열 DTC 코드는?",    "표 포함 → 키워드 검색 중요"),
    ("SOC 추정 방법을 설명해줘",         "의미 검색"),
    ("고전압 작업 안전 절차",            "의미 검색"),
]

print("=== 하이브리드 검색 결과 ===\n")
for query, note in test_cases:
    print(f"쿼리: '{query}'  ({note})")
    results = hybrid_search_preview(query, top_k=3)
    for rank, (cid, score) in enumerate(results):
        print(f"  #{rank+1} [{cid}] hybrid_score={score:.3f}")
    print()

In [ ]:
# BM25 인덱스 직렬화 저장 (NB03 로드용)
import pickle

bm25_save = {
    'bm25':            bm25,
    'tokenized_corpus': tokenized_corpus,
    'chunk_ids':       [c['chunk_id'] for c in chunks]
}
bm25_path = OUTPUT_DIR / 'bm25_index.pkl'
with open(bm25_path, 'wb') as f:
    pickle.dump(bm25_save, f)

print(f"BM25 인덱스 저장: {bm25_path}")
print("\n=== NB02 완료 ===")
print(f"  BGE-M3 임베딩: {embeddings.shape}")
print(f"  ChromaDB: {collection.count()}개 문서 인덱싱")
print(f"  BM25 인덱스: {len(tokenized_corpus)}개 문서")
print("\n[다음 단계] Notebook 03 — RAG 파이프라인 (하이브리드 검색 + Qwen2.5-VL 답변 생성)")